In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error
from src.custom_fastkan import FastKAN
import pandas as pd

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [2]:
def true_function(x):
    # f(x, y) = exp(sin(pi*x) + y^2)
    return torch.exp(torch.sin(torch.pi * x[:, 0]) + x[:, 1]**2)

torch.manual_seed(42)
num_samples = 10000

X = torch.rand(num_samples, 2) * 2 - 1 
y = true_function(X)

train_size = int(0.7 * num_samples)
val_size = int(0.15 * num_samples)
test_size = num_samples - train_size - val_size

X_train, X_val, X_test = torch.split(X, [train_size, val_size, test_size])
y_train, y_val, y_test = torch.split(y, [train_size, val_size, test_size])

X_train = X_train.to(device)
y_train = y_train.to(device)
X_val = X_val.to(device)
y_val = y_val.to(device)
X_test = X_test.to(device)
y_test = y_test.to(device)

print(f"Train shape: {X_train.shape}, Val shape: {X_val.shape}, Test shape: {X_test.shape}")

Train shape: torch.Size([7000, 2]), Val shape: torch.Size([1500, 2]), Test shape: torch.Size([1500, 2])


In [3]:
model = FastKAN([2, 1, 1], grid_min=-3, grid_max=3, num_grids=10, use_base_update=False, use_layernorm=False).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [4]:
print("Training KAN...")
train_losses = []
val_losses = []

for epoch in range(1000):
    optimizer.zero_grad()
    pred = model(X_train).squeeze()
    loss = torch.mean((pred - y_train)**2)
    loss.backward()
    optimizer.step()
    
    if epoch % 50 == 0:
        with torch.no_grad():
            val_pred = model(X_val).squeeze()
            val_loss = torch.mean((val_pred - y_val)**2)
            train_losses.append(loss.item())
            val_losses.append(val_loss.item())
            print(f"Epoch {epoch}, Train MSE: {loss.item():.6f}, Val MSE: {val_loss.item():.6f}")

model.eval()
with torch.no_grad():
    test_pred = model(X_test).squeeze()
    mse = mean_squared_error(y_test.cpu(), test_pred.cpu())
    r2 = r2_score(y_test.cpu(), test_pred.cpu())
    
    print(f"KAN Test MSE: {mse:.6f}")
    print(f"KAN Test R2: {r2:.4f}")

Training KAN...
Epoch 0, Train MSE: 5.701476, Val MSE: 5.642240
Epoch 50, Train MSE: 3.185184, Val MSE: 3.184538
Epoch 100, Train MSE: 1.110303, Val MSE: 1.153340
Epoch 150, Train MSE: 0.429348, Val MSE: 0.475965
Epoch 200, Train MSE: 0.194728, Val MSE: 0.222409
Epoch 250, Train MSE: 0.115200, Val MSE: 0.133819
Epoch 300, Train MSE: 0.080654, Val MSE: 0.093923
Epoch 350, Train MSE: 0.058015, Val MSE: 0.067887
Epoch 400, Train MSE: 0.039795, Val MSE: 0.047046
Epoch 450, Train MSE: 0.025001, Val MSE: 0.029649
Epoch 500, Train MSE: 0.015115, Val MSE: 0.017905
Epoch 550, Train MSE: 0.010403, Val MSE: 0.012380
Epoch 600, Train MSE: 0.007847, Val MSE: 0.009354
Epoch 650, Train MSE: 0.006274, Val MSE: 0.007451
Epoch 700, Train MSE: 0.005233, Val MSE: 0.006168
Epoch 750, Train MSE: 0.004492, Val MSE: 0.005245
Epoch 800, Train MSE: 0.003929, Val MSE: 0.004542
Epoch 850, Train MSE: 0.003481, Val MSE: 0.003984
Epoch 900, Train MSE: 0.003112, Val MSE: 0.003529
Epoch 950, Train MSE: 0.002802, Val M

In [5]:
import torch.nn as nn
torch.manual_seed(42)
class MLP(nn.Module):
    def __init__(self, input_dim=2, hidden_dims=[64, 64], output_dim=1):
        super(MLP, self).__init__()
        layers = []
        curr_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(curr_dim, h_dim))
            layers.append(nn.ReLU())
            curr_dim = h_dim
        layers.append(nn.Linear(curr_dim, output_dim))
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)

mlp_model = MLP(input_dim=2, hidden_dims=[32, 32], output_dim=1).to(device)
mlp_optimizer = torch.optim.Adam(mlp_model.parameters(), lr=0.01)

print("Training MLP...")
for epoch in range(1000):
    mlp_model.train()
    mlp_optimizer.zero_grad()
    pred = mlp_model(X_train).squeeze()
    loss = torch.mean((pred - y_train)**2)
    loss.backward()
    mlp_optimizer.step()
    
    if epoch % 100 == 0:
        mlp_model.eval()
        with torch.no_grad():
            val_pred = mlp_model(X_val).squeeze()
            val_loss = torch.mean((val_pred - y_val)**2)
            print(f"Epoch {epoch}, Train MSE: {loss.item():.6f}, Val MSE: {val_loss.item():.6f}")

mlp_model.eval()
with torch.no_grad():
    mlp_pred = mlp_model(X_test).squeeze()
    mlp_mse = mean_squared_error(y_test.cpu(), mlp_pred.cpu())
    mlp_r2 = r2_score(y_test.cpu(), mlp_pred.cpu())
    
    print(f"\nMLP Test MSE: {mlp_mse:.6f}")
    print(f"MLP Test R2: {mlp_r2:.4f}")

Training MLP...
Epoch 0, Train MSE: 5.587737, Val MSE: 4.983770
Epoch 100, Train MSE: 0.633662, Val MSE: 0.704734
Epoch 200, Train MSE: 0.038329, Val MSE: 0.037183
Epoch 300, Train MSE: 0.009997, Val MSE: 0.009952
Epoch 400, Train MSE: 0.007894, Val MSE: 0.007864
Epoch 500, Train MSE: 0.006484, Val MSE: 0.006557
Epoch 600, Train MSE: 0.004840, Val MSE: 0.005338
Epoch 700, Train MSE: 0.003351, Val MSE: 0.003785
Epoch 800, Train MSE: 0.002823, Val MSE: 0.003167
Epoch 900, Train MSE: 0.002624, Val MSE: 0.002892

MLP Test MSE: 0.002493
MLP Test R2: 0.9988


In [6]:
print("\nAnalyzing individual KAN prediction losses...")
individual_losses = []
predictions = []
with torch.no_grad():
    for i in range(len(X_test)):
        input_seq = X_test[i]
        ground_truth = y_test[i]
        prediction = model(input_seq.unsqueeze(0)).squeeze()
        
        loss = ((prediction - ground_truth) ** 2).item()
        individual_losses.append(loss)
        predictions.append(prediction)

individual_losses = np.array(individual_losses)
predictions = torch.stack(predictions)
sorted_indices = np.argsort(individual_losses)

mean_loss = np.mean(individual_losses)
lowest_indices = sorted_indices[:3] 
highest_indices = sorted_indices[-3:]

mean_distances = np.abs(individual_losses - mean_loss)
mean_sorted_indices = np.argsort(mean_distances)
mean_indices = mean_sorted_indices[:3]

print(f"\nKAN Loss Statistics:")
print(f"Mean Loss: {mean_loss:.6f}")
print(f"Min Loss: {individual_losses[lowest_indices[0]]:.6f}")
print(f"Max Loss: {individual_losses[highest_indices[-1]]:.6f}")


Analyzing individual KAN prediction losses...

KAN Loss Statistics:
Mean Loss: 0.002797
Min Loss: 0.000000
Max Loss: 0.418829


In [7]:
print("\nAnalyzing individual MLP prediction losses...")
mlp_individual_losses = []
mlp_predictions = []

mlp_model.eval()
with torch.no_grad():
    for i in range(len(X_test)):
        input_seq = X_test[i]
        ground_truth = y_test[i]
        prediction = mlp_model(input_seq.unsqueeze(0)).squeeze()
        
        loss = ((prediction - ground_truth) ** 2).item()
        mlp_individual_losses.append(loss)
        mlp_predictions.append(prediction)

mlp_individual_losses = np.array(mlp_individual_losses)
mlp_predictions = torch.stack(mlp_predictions)
mlp_sorted_indices = np.argsort(mlp_individual_losses)

mlp_mean_loss = np.mean(mlp_individual_losses)
mlp_lowest_indices = mlp_sorted_indices[:3] 
mlp_highest_indices = mlp_sorted_indices[-3:]

mlp_mean_distances = np.abs(mlp_individual_losses - mlp_mean_loss)
mlp_mean_sorted_indices = np.argsort(mlp_mean_distances)
mlp_mean_indices = mlp_mean_sorted_indices[:3]

print(f"\nMLP Loss Statistics:")
print(f"Mean Loss: {mlp_mean_loss:.6f}")
print(f"Min Loss: {mlp_individual_losses[mlp_lowest_indices[0]]:.6f}")
print(f"Max Loss: {mlp_individual_losses[mlp_highest_indices[-1]]:.6f}")


Analyzing individual MLP prediction losses...

MLP Loss Statistics:
Mean Loss: 0.002493
Min Loss: 0.000000
Max Loss: 0.118380


In [ ]:
table_data = []
categories = [("Lowest", lowest_indices), ("Highest", highest_indices), ("Mean", mean_indices)]

for label, indices in categories:
    for idx in indices:
        table_data.append({
            "Category": label,
            "Index": idx,
            "Input (x, y)": f"({X_test[idx][0].item():.4f}, {X_test[idx][1].item():.4f})",
            "True Value": y_test[idx].item(),
            "Predicted": predictions[idx].item(),
            "Loss": individual_losses[idx]
        })

df_kan_analysis = pd.DataFrame(table_data)
df_kan_analysis

,Category,Index,"Input (x, y)",True Value,Predicted,Loss
0,Lowest,368,"(-0.6707, -0.7744)",0.771142,0.771168,6.722622e-10
1,Lowest,1030,"(-0.9235, 0.5392)",1.054009,1.054102,8.535395e-09
2,Lowest,982,"(-0.5379, 0.9854)",0.978325,0.978220,1.109260e-08
3,Highest,177,"(0.4857, 0.9817)",7.118301,6.548188,3.250290e-01
4,Highest,1373,"(0.4507, 0.9907)",7.167521,6.573806,3.524977e-01
5,Highest,615,"(0.5354, -0.9938)",7.253314,6.606143,4.188291e-01
6,Mean,323,"(0.0424, 0.9091)",2.609769,2.556947,2.790151e-03
7,Mean,1150,"(0.1131, 0.7846)",2.620998,2.568043,2.804197e-03
8,Mean,1382,"(0.1112, 0.7673)",2.536820,2.483575,2.835063e-03


In [9]:
table_data_mlp = []
categories = [("Lowest", mlp_lowest_indices), ("Highest", mlp_highest_indices), ("Mean", mlp_mean_indices)]

for label, indices in categories:
    for idx in indices:
        table_data_mlp.append({
            "Category": label,
            "Index": idx,
            "Input (x, y)": f"({X_test[idx][0].item():.4f}, {X_test[idx][1].item():.4f})",
            "True Value": y_test[idx].item(),
            "Predicted": mlp_predictions[idx].item(),
            "Loss": mlp_individual_losses[idx]
        })

df_mlp_analysis = pd.DataFrame(table_data_mlp)
df_mlp_analysis

,Category,Index,"Input (x, y)",True Value,Predicted,Loss
0,Lowest,253,"(-0.1311, -0.9675)",1.708874,1.708911,1.356867e-09
1,Lowest,1234,"(-0.9175, 0.4251)",0.927178,0.927243,4.197777e-09
2,Lowest,643,"(0.8714, 0.1778)",1.529242,1.529314,5.201528e-09
3,Highest,691,"(-0.9906, 0.9536)",2.410768,2.135784,7.561640e-02
4,Highest,615,"(0.5354, -0.9938)",7.253314,6.940802,9.766341e-02
5,Highest,493,"(-0.9972, 0.9704)",2.542152,2.198088,1.183797e-01
6,Mean,574,"(0.4394, -0.9274)",6.309407,6.259479,2.492872e-03
7,Mean,1161,"(-0.4381, 0.6427)",0.566600,0.616524,2.492449e-03
8,Mean,125,"(0.4984, 0.4785)",3.417724,3.367782,2.494134e-03


In [10]:
mlp_save_path = "model_pkls/functionexp_mlp_model.pkl"
torch.save({
    'model_state_dict': mlp_model.state_dict(),
    'config': {
        'input_dim': 2,
        'hidden_dims': [32, 32, 32],
        'output_dim': 1
    }
}, mlp_save_path)
print(f"MLP Model saved to {mlp_save_path}")

MLP Model saved to model_pkls/functionexp_mlp_model.pkl


In [11]:
kan_save_path = "model_pkls/functionexp_kan_model.pkl"
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'layers_hidden': [2, 1, 1],
        'grid_min': -3,
        'grid_max': 3,
        'num_grids': 10,
        'use_base_update': False,
        'use_layernorm': False,
    }
}, kan_save_path)
print(f"KAN Model saved to {kan_save_path}")

KAN Model saved to model_pkls/functionexp_kan_model.pkl
